# Hannah's notebook

## Embeddingscode uit clustering-notebook

In [ ]:
import glob
import shutil
import threading
import warnings
from pathlib import Path

import numpy as np
import torch

from embed import (
    CLIPEmbedder,
    get_device,
    save_embeddings,
)

In [ ]:
MODEL = "laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K"
PROJECT_ROOT = Path('project_root')  # Vervang door het pad naar de root van je project
PACKAGE_ROOT = PROJECT_ROOT / "cluster"
EMBEDDINGS_PATH = PACKAGE_ROOT / "embeddings_file.pkl"
IMAGE_DIR = Path("/path/to/image/directory")  # Vervang door het pad naar de directory met je afbeeldingen

In [ ]:
import transformers

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=UserWarning)

_orig_excepthook = threading.excepthook

def _excepthook(args):
    if args.thread is not None and "auto_conversion" in args.thread.name:
        return
    _orig_excepthook(args)


threading.excepthook = _excepthook

In [ ]:
device = get_device()
embedder = CLIPEmbedder(MODEL)
embedder.load(device)

In [ ]:
image_paths = sorted(glob.glob(f"{IMAGE_DIR}/*.jpg"))

In [ ]:
embeddings, image_paths = embedder.encode_images(image_paths, batch_size=8)

In [ ]:
#optioneel: data opslaan in bestand

save_embeddings(
    "./embeddings_file.pkl",
    image_paths,
    embeddings,
    embedder,
)

In [ ]:
# om de data terug uit het bestand te halen, ipv opnieuw te embedden

from embed import load_embeddings

data = load_embeddings(
    "./embeddings_file.pkl"
)

embeddings = data["embeddings"]
image_paths = data["images"]

print(type(embeddings))
print(type(image_paths))

## Artwork en niet-artwork definiëren en embeddings vergelijken

In [ ]:
artwork_prompts = [
    "photograph of an artwork",
    "photograph of a painting",
    "photograph of a drawing",
    "photograph of a sculpture",
    "image showing a work of art",
    "photograph of a museum artwork",
    "an artwork",
    "photograph of a framed painting",
    "photograph of an artistic object",
    "photograph of a religious artwork"
]

non_artwork_prompts = [
    "normal photograph with no artwork",
    "real-world scene",
    "photograph of a person, object, or place",
    "image that does not contain a painting or drawing",
    "a building",
    "regular documentary photograph",
    "photograph of an everyday object",
    "photograph of a natural scene",
    "photograph of a building or location",
    "ordinary photo"
]

In [ ]:
all_prompts = artwork_prompts + non_artwork_prompts

text_inputs = embedder._processor(
    text=all_prompts,
    return_tensors="pt",
    padding=True,
)

# Prompts converteren naar input voor het model

text_inputs = {
    key: value.to(embedder._device)
    for key, value in text_inputs.items()
}

# CLIP-model genereert embeddings voor de prompts

with torch.inference_mode():
    text_output = embedder._model.get_text_features(
        **text_inputs
    )

if hasattr(text_output, "pooler_output"):
    text_embeddings = text_output.pooler_output
else:
    text_embeddings = text_output

# Converteert de embeddings naar numpy array

text_embeddings = text_embeddings.detach().cpu().numpy()

In [ ]:
#Definieer de 2 categorieën

number_of_artwork_prompts = len(artwork_prompts)

artwork_embedding = text_embeddings[
    :number_of_artwork_prompts
].mean(axis=0)

non_artwork_embedding = text_embeddings[
    number_of_artwork_prompts:
].mean(axis=0)

In [ ]:
#Normalizeer de embeddings zodat ze een lengte van 1 hebben

def normalize_rows(x):
    return x / np.linalg.norm(
        x,
        axis=1,
        keepdims=True,
    )

image_embeddings = normalize_rows(embeddings)

text_embeddings = normalize_rows(
    np.vstack(
        [
            artwork_embedding,
            non_artwork_embedding,
        ]
    )
)

In [ ]:
#Vergelijk de foto's met de prompts en bereken de cosine similarity
# foto → [artwork similarity, non-artwork similarity]

similarities = image_embeddings @ text_embeddings.T

artwork_scores = similarities[:, 0]
non_artwork_scores = similarities[:, 1]

is_artwork = artwork_scores > non_artwork_scores

In [ ]:
print(f"Artwork: {is_artwork.sum()}")
print(f"Non-artwork: {(~is_artwork).sum()}")
print(f"Check: {(~is_artwork).sum()+(is_artwork.sum())}")

In [ ]:
CLUSTER_FILE_PATH = Path("cluster/path") # Vervang door het pad naar de directory waar je de mappen wilt opslaan

ARTWORK_DIR = CLUSTER_FILE_PATH / "artwork"
NON_ARTWORK_DIR = CLUSTER_FILE_PATH / "non-artwork"

ARTWORK_DIR.mkdir(parents=True, exist_ok=True)
NON_ARTWORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Optioneel: verander de file paths van de foto's in de embeddingsfile als de image directory is veranderd (door bijvoorbeeld het mounten van een harde schijf)

current_files_by_name = {
    path.name: path
    for path in IMAGE_DIR.glob("*.jpg")
}
updated_image_paths = []

for old_path in image_paths:
    filename = Path(old_path).name
    new_path = current_files_by_name.get(filename)
    updated_image_paths.append(new_path)

for old_path, new_path in zip( image_paths[:5], updated_image_paths[:5], strict=False, ):
    print("Oud: ", old_path)
    print("Nieuw:", new_path)
    print()

In [ ]:
# steekproef

# for new_path, artwork in zip(updated_image_paths[:100], is_artwork[:100]):
#     source_path = Path(new_path)

#     if artwork:
#         destination_dir = ARTWORK_DIR
#     else:
#         destination_dir = NON_ARTWORK_DIR

#     shutil.copy2(
#         source_path,
#         destination_dir / source_path.name,
#     )

In [ ]:
# De foto's in de clusteringsmappen kopiëren

for new_path, artwork in zip(updated_image_paths, is_artwork, strict=False):
    source_path = Path(new_path)

    if artwork:
        destination_dir = ARTWORK_DIR
    else:
        destination_dir = NON_ARTWORK_DIR

    shutil.copy2(
        source_path,
        destination_dir / source_path.name,
    )

## Sample nemen

In [ ]:
# Functie om een steekproef te maken van de embeddingsfile met sample_size als grootte van de steekproef

import pickle
import random
from pathlib import Path


def make_sample(input_pickle, sample_size):

    input_pickle = Path(input_pickle)

    with input_pickle.open("rb") as file:
        data = pickle.load(file)

    image_paths = data["images"]
    embeddings = data["embeddings"]

    indices = random.sample(range(len(image_paths)), sample_size)

    sampled_data = {
        "images": [image_paths[i] for i in indices],
        "embeddings": embeddings[indices] if hasattr(embeddings, "__getitem__") else [
            embeddings[i] for i in indices
        ],
    }

    output_pickle = input_pickle.with_name(
        f"sample_{sample_size}.pkl"
    )

    with output_pickle.open("wb") as file:
        pickle.dump(sampled_data, file)

    return output_pickle

In [ ]:
make_sample(
    input_pickle="./embeddings_file.pkl", # Vervang door het pad naar de embeddingsfile
    sample_size=3595, # Vervang door de gewenste grootte van de steekproef
)

## Extra functies

In [ ]:
# Functie om de referentie-afbeeldingen in de embeddingsfile te filteren en bij te werken op basis van een nieuwe image directory

from copy import deepcopy
from pathlib import Path


def filter_and_update_embeddings(
    embeddings_pickle,
    image_dir
    ):
    embeddings_pickle = Path(embeddings_pickle)
    image_dir = Path(image_dir)
    name = image_dir.name

    with embeddings_pickle.open("rb") as file:
        data = pickle.load(file)

    old_paths = data["images"]
    embeddings = data["embeddings"]

    current_images = {
    image.name: image
    for image in image_dir.rglob("*")
}

    new_data = deepcopy(data)

    matched_paths = []
    matched_indices = []

    for index, old_path in enumerate(old_paths):
        filename = Path(old_path).name

        if filename in current_images:
            matched_indices.append(index)
            matched_paths.append(str(current_images[filename].resolve()))

    filtered_embeddings = embeddings[matched_indices]

    new_data["images"] = matched_paths
    new_data["embeddings"] = filtered_embeddings

    output_pickle = embeddings_pickle.with_name(
        f"embeddings_{name}.pkl"
    )
    with output_pickle.open("wb") as file:
        pickle.dump(new_data, file)

    return new_data

In [ ]:
filter_and_update_embeddings("./embeddings_file.pkl", "./image_directory")

In [ ]:
# Functie om enkel de afbeeldingen die in de embeddingsfile voorkomen, te kopiëren naar een eigen map

import shutil
from pathlib import Path


def copy_images_from_pickle(pickle_path, output_dir):
    pickle_path = Path(pickle_path)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    with open(pickle_path, "rb") as file:
        data = pickle.load(file)

    image_paths = data["images"]

    for image_path in image_paths:
        source_path = Path(image_path)
        destination_path = output_dir / source_path.name
        shutil.copy2(source_path, destination_path)

In [ ]:
copy_images_from_pickle("./embeddings_file.pkl", "path/to/output/directory")

In [ ]:
# Functie om een embeddingsfile op te splitsen in meerdere bestanden

import math
from pathlib import Path


def split_embeddings_file(input_file, output_dir, number):

    input_file = Path(input_file)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with input_file.open("rb") as file:
        data = pickle.load(file)

    image_paths = data["images"]
    embeddings = data["embeddings"]

    total_items = len(image_paths)

    chunk_size = math.ceil(total_items/number)

    output_files = []

    for part_number, start in enumerate(
        range(0, total_items, chunk_size),
        start=1
    ):
        end = min(start + chunk_size, total_items)

        part_data = data.copy()
        part_data["images"] = image_paths[start:end]
        part_data["embeddings"] = embeddings[start:end]

        output_file = output_dir / (
            f"{input_file.stem}_part_{part_number:03d}.pkl"
        )

        with output_file.open("wb") as file:
            pickle.dump(part_data, file)

        output_files.append(output_file)

    return output_files

In [ ]:
split_embeddings_file("./embeddings_file.pkl", "path/to/output/directory", 20)

In [ ]:
#  Functie om dubbels uit de embeddingsfile verwijderen

import json


def delete_double_images(json_file):
    with open(json_file) as f:
        data = json.load(f)

    seen = set()
    unique_data = {}

    for key, entries in data.items():
        unique_entries = []
        for entry in entries:
            glashelder_index = entry['glashelder_index']
            if glashelder_index not in seen:
                seen.add(glashelder_index)
                unique_entries.append(entry)
        unique_data[key] = unique_entries

    with open(json_file, 'w') as f:
        json.dump(unique_data, f, indent=4)

In [ ]:
delete_double_images('./dataset_met_dubbels.json')

## Excel

In [ ]:
#Json bekijken om structuur te achterhalen

with open('/path_to/file.json') as f:
    x = json.load(f)
print(json.dumps(x, indent=5))

In [ ]:
# Omzetten van json naar excel, waarbij elke key in de json een aparte sheet in de excel file wordt

import json

import pandas as pd

with open("./labeling_dataset.json") as f:
  data = json.load(f)

with pd.ExcelWriter("./labels.xlsx", engine="openpyxl") as writer:
  for key, entries in data.items():
    df = pd.DataFrame(entries)
    df.to_excel(writer, sheet_name=key, index=False)